# 🎓 Cost and Parts Prediction Model Training
This notebook trains the `RandomForestRegressor` and `RandomForestClassifier` for the cost prediction module.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
import joblib

### 1. Load the Dataset
We load the historical repair data to train the model. This gives the AI examples of how much past repairs cost.

In [ ]:
print("Loading dataset...")
# Note: path is relative to the model folder
df = pd.read_excel("cleaned_parts_required_dataset.xlsx")
df = df.dropna()
df.head() # This will display a preview of the dataset

### 2. Define Features and Targets
Here we define our Inputs (X) and our two Outputs (y_parts, y_cost).

In [ ]:
X = df[['Device_Type', 'Item_Model', 'Fault_Description']]
y_parts = df['Parts_Required']
y_cost = df['Cost']

# Split 80% of data for training, and 20% for testing
X_train, X_test, y_parts_train, y_parts_test = train_test_split(X, y_parts, test_size=0.2, random_state=42)
_, _, y_cost_train, y_cost_test = train_test_split(X, y_cost, test_size=0.2, random_state=42)

print("Data successfully split into training and testing sets.")

### 3. Preprocessing (One-Hot Encoding)
Machine learning models require numerical input. We use a `OneHotEncoder` to encode our text categories into binary numbers.

In [ ]:
preprocessor = ColumnTransformer(
    [
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Device_Type', 'Item_Model', 'Fault_Description'])
    ]
)

### 4. Train the Parts Model (Classifier)
We use a `RandomForestClassifier` because parts are categorical.

In [ ]:
print("Training Parts Model with Random Forest Classifier (200 trees)...")
parts_model = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])
parts_model.fit(X_train, y_parts_train)

# Evaluate Parts Model
pred_parts = parts_model.predict(X_test)
acc = accuracy_score(y_parts_test, pred_parts)
print(f"Parts Model Trained! Accuracy: {acc * 100:.2f}%")

### 5. Train the Cost Model (Regressor)
We use a `RandomForestRegressor` because cost is a continuous numerical value.

In [ ]:
print("Training Cost Model with Random Forest Regressor (200 trees)...\n")
cost_model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])
cost_model.fit(X_train, y_cost_train)

# Evaluate Cost Model
pred_cost = cost_model.predict(X_test)
mae = mean_absolute_error(y_cost_test, pred_cost)
r2 = r2_score(y_cost_test, pred_cost)
print(f"Cost Model Trained! R? Score (Accuracy): {r2 * 100:.2f}% | Average Error: Rs. {mae:.2f}")

### 6. Save the Models
Export the trained models to `.pkl` files so the PHP API can use them.

In [ ]:
# We save them in the time_prediction_project folder for the API to access
joblib.dump(parts_model, "parts_model.pkl")
joblib.dump(cost_model, "cost_model.pkl")
print("Models saved successfully to parts_model.pkl and cost_model.pkl!")

### 7. Live Example Prediction
Let's test the AI right now! We will input a completely new repair job (Laptop, Dell XPS 15, No Power) and ask the models to predict the required parts and final cost.

In [ ]:
# Create a brand new repair ticket
sample_repair = pd.DataFrame([{
    'Device_Type': 'Laptop',
    'Item_Model': 'Dell XPS 15',
    'Fault_Description': 'No Power'
}])

print(f"New Repair Ticket Incoming: {sample_repair['Device_Type'][0]} - {sample_repair['Item_Model'][0]} ({sample_repair['Fault_Description'][0]})\n")

# Ask the models to predict
predicted_parts = parts_model.predict(sample_repair)[0]
predicted_cost = cost_model.predict(sample_repair)[0]

# Print the final AI Prediction
print("---- AI PREDICTION ----")
print(f"Required Parts: {predicted_parts}")
print(f"Estimated Cost: Rs. {predicted_cost:,.2f}")